# Download WV2 L1B for SERC and GPNF

Searches NASA CMR for WorldView-2 8-band multispectral Level 1B granules intersecting each site. For each site independently, it tries July 2025 first, then July of each previous year until data are found or `EARLIEST_YEAR` is reached. It downloads the granule NTF and available metadata/sidecars from CSDA's S3 bucket and writes a CSV manifest.

SERC uses the site extent from the previous notebook. GPNF is treated as Gifford Pinchot National Forest; its bounding box below is an approximate regional extent because no site AOI file was provided. Replace it with an authoritative AOI if available. S3 download requires CSDA temporary S3 credentials and an environment authorized for the CSDA bucket (normally AWS us-west-2).

In [1]:
from __future__ import annotations

import csv
import getpass
import html.parser
import json
import posixpath
import re
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

import boto3
import requests
from botocore.config import Config

CMR_GRANULES_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"
S3_CREDENTIALS_URL = "https://data.csdap.earthdata.nasa.gov/s3credentials"
WV2_L1B_COLLECTION_ID = "C2497404794-CSDA"  # WV02_MSI_L1B
START_YEAR = 2025
EARLIEST_YEAR = 2009  # WV2 archive begins in 2009; adjust the search floor if needed.
MONTH = 7
PAGE_SIZE = 2000
OUTPUT_DIR = Path("/explore/nobackup/people/pmontesa/WV2_L1B_July")
MANIFEST_PATH = OUTPUT_DIR / "wv2_l1b_july_site_downloads.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CMR bounding boxes are [west, south, east, north] in EPSG:4326.
SITES = {
    "SERC": (-76.736, 38.740, -76.386, 39.040),
    "GPNF": (-122.50, 45.50, -121.30, 46.50),
}

DOWNLOAD_SUFFIXES = {".ntf", ".tif", ".tiff", ".xml", ".rpb", ".imd"}

## 1. Search July data with per-site year fallback
The first year with one or more matching WV2 L1B granules is selected independently for each site. Search errors stop the notebook rather than being mistaken for an empty year.

In [2]:
def july_window(year: int) -> str:
    return f"{year}-07-01T00:00:00Z,{year}-07-31T23:59:59.999Z"


def search_site_year(site_name: str, bbox: tuple[float, float, float, float], year: int) -> list[dict]:
    granules = []
    page_num = 1
    session = requests.Session()
    while True:
        params = {
            "collection_concept_id": WV2_L1B_COLLECTION_ID,
            "page_size": PAGE_SIZE,
            "page_num": page_num,
            "bounding_box": ",".join(map(str, bbox)),
            "temporal": july_window(year),
        }
        response = session.get(CMR_GRANULES_URL, params=params, timeout=60)
        response.raise_for_status()
        entries = response.json().get("feed", {}).get("entry", [])
        if not entries:
            break
        granules.extend(entries)
        hits = int(response.headers.get("CMR-Hits", len(granules)))
        if len(granules) >= hits or len(entries) < PAGE_SIZE:
            break
        page_num += 1
        time.sleep(0.2)
    return granules


selected_by_site = {}
for site_name, bbox in SITES.items():
    print(f"\nSearching {site_name} for July {START_YEAR}")
    for year in range(START_YEAR, EARLIEST_YEAR - 1, -1):
        granules = search_site_year(site_name, bbox, year)
        print(f"  July {year}: {len(granules)} matching granule(s)")
        if granules:
            selected_by_site[site_name] = {"year": year, "granules": granules}
            break
    else:
        selected_by_site[site_name] = {"year": None, "granules": []}
        print(f"  No July granules found back to {EARLIEST_YEAR}.")

for site_name, result in selected_by_site.items():
    if result["year"] is not None:
        print(f"{site_name}: using July {result['year']} ({len(result['granules'])} granules)")
    else:
        print(f"{site_name}: no data found")


Searching SERC for July 2025
  July 2025: 0 matching granule(s)
  July 2024: 9 matching granule(s)

Searching GPNF for July 2025
  July 2025: 0 matching granule(s)
  July 2024: 7 matching granule(s)
SERC: using July 2024 (9 granules)
GPNF: using July 2024 (7 granules)


## 2. Authenticate for temporary CSDA S3 access
Provide your CSDA Cognito login and authenticator MFA code. Credentials are temporary and are not written to disk; the downloader will prompt again if they are near expiration.

In [3]:
class InputParser(html.parser.HTMLParser):
    def __init__(self):
        super().__init__()
        self.values = {}

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "input":
            attributes = dict(attrs)
            name = attributes.get("name")
            if name:
                self.values[name] = attributes.get("value", "")


def fetch_s3_credentials():
    username = input("CSDA Cognito username: ")
    password = getpass.getpass("CSDA Cognito password: ")
    session = requests.Session()

    login_page = session.get(S3_CREDENTIALS_URL, timeout=60)
    login_page.raise_for_status()
    form = InputParser()
    form.feed(login_page.text)
    if "_csrf" not in form.values:
        raise RuntimeError("Could not find CSRF token on the CSDA login page")

    login_data = {
        "_csrf": form.values["_csrf"],
        "username": username,
        "password": password,
    }
    if form.values.get("cognitoAsfData"):
        login_data["cognitoAsfData"] = form.values["cognitoAsfData"]
    login_page = session.post(login_page.url, data=login_data, timeout=60)
    login_page.raise_for_status()

    mfa_form = InputParser()
    mfa_form.feed(login_page.text)
    if "_csrf" not in mfa_form.values:
        raise RuntimeError("CSDA did not present an MFA form; inspect the login response")
    mfa_data = {
        "_csrf": mfa_form.values["_csrf"],
        "authentication_code": input("CSDA authenticator MFA code: ").strip(),
        "cognitoAsfData": mfa_form.values.get("cognitoAsfData", ""),
    }
    response = session.post(
        login_page.url, data=mfa_data, allow_redirects=False, timeout=60
    )
    if response.is_redirect:
        response = session.get(urljoin(response.url, response.headers["Location"]), timeout=60)
    response.raise_for_status()
    if "json" not in response.headers.get("content-type", "").lower():
        response = session.get(S3_CREDENTIALS_URL, timeout=60)
        response.raise_for_status()
    credentials = response.json()
    required = {"accessKeyId", "secretAccessKey", "sessionToken"}
    if not required.issubset(credentials):
        raise RuntimeError("CSDA response did not contain the expected temporary S3 credentials")
    return credentials


def build_s3_client(credentials):
    return boto3.client(
        "s3",
        aws_access_key_id=credentials["accessKeyId"],
        aws_secret_access_key=credentials["secretAccessKey"],
        aws_session_token=credentials["sessionToken"],
        region_name="us-west-2",
        config=Config(retries={"max_attempts": 5, "mode": "standard"}),
    )


def refresh_s3_client():
    global credentials, s3_client, credential_expiration
    credentials = fetch_s3_credentials()
    expiration = credentials.get("expiration")
    credential_expiration = (
        datetime.fromisoformat(expiration.replace("Z", "+00:00"))
        if expiration else datetime.now(timezone.utc) + timedelta(minutes=55)
    )
    if credential_expiration.tzinfo is None:
        credential_expiration = credential_expiration.replace(tzinfo=timezone.utc)
    s3_client = build_s3_client(credentials)


def ensure_s3_client():
    if credential_expiration <= datetime.now(timezone.utc) + timedelta(minutes=5):
        print("Temporary credentials are expiring; authenticate again to continue downloads.")
        refresh_s3_client()
    return s3_client


refresh_s3_client()
print("Temporary CSDA S3 credentials acquired; credentials are short-lived.")

CSDA Cognito username:  montesano
CSDA Cognito password:  ········
CSDA authenticator MFA code:  688975


Temporary CSDA S3 credentials acquired; credentials are short-lived.


## 3. Download granules and save manifest
Downloads NTF/TIFF imagery and available XML/RPB/IMD sidecars into site/year/granule folders.

In [9]:
def s3_data_links(granule: dict) -> list[tuple[str, str]]:
    'Return S3 bucket/key pairs for NTF data and available sidecars.'
    files = {}
    for link in granule.get("links", []):
        href = link.get("href", "")
        if not href.startswith("s3://"):
            continue
        parsed = urlparse(href)
        suffix = Path(parsed.path).suffix.lower()
        if suffix not in DOWNLOAD_SUFFIXES:
            continue
        key = parsed.path.lstrip("/")
        if parsed.netloc and key:
            files[(parsed.netloc, key)] = (parsed.netloc, key)
    return list(files.values())


def safe_name(value: str) -> str:
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._")
    return name or "granule"


def download_site_granules() -> list[dict]:
    manifest_rows = []
    for site_name, result in selected_by_site.items():
        year = result["year"]
        if year is None:
            continue
        for granule in result["granules"]:
            granule_id = granule.get("producer_granule_id") or granule.get("title") or granule.get("id")
            granule_dir = OUTPUT_DIR / site_name / str(year) / safe_name(granule_id)
            granule_dir.mkdir(parents=True, exist_ok=True)
            links = s3_data_links(granule)
            if not any(Path(key).suffix.lower() in {".ntf", ".tif", ".tiff"} for _, key in links):
                raise RuntimeError(
                    f"No NTF/TIFF S3 data link found for CMR granule {granule_id}; "
                    "refusing to report this granule as downloaded."
                )

            local_files = []
            for bucket, key in links:
                filename = safe_name(posixpath.basename(key))
                destination = granule_dir / filename
                if destination.exists() and destination.stat().st_size > 0:
                    print(f"Exists: {destination}")
                else:
                    print(f"Downloading s3://{bucket}/{key}")
                    for attempt in range(2):
                        try:
                            ensure_s3_client().download_file(bucket, key, str(destination))
                            break
                        except Exception as error:
                            error_response = getattr(error, "response", {})
                            aws_error = error_response.get("Error", {})
                            error_code = aws_error.get("Code", type(error).__name__)
                            error_message = aws_error.get("Message", str(error))
                            expired_token_codes = {
                                "ExpiredToken", "ExpiredTokenException",
                                "InvalidToken", "TokenRefreshRequired",
                            }
                            if attempt == 0 and error_code in expired_token_codes:
                                print("S3 token expired; refreshing credentials and retrying once.")
                                refresh_s3_client()
                                continue

                            if error_code in {"AccessDenied", "403", "AllAccessDisabled"}:
                                advice = (
                                    " The AWS principal is not authorized by the CSDA bucket policy. "
                                    "Setting region_name='us-west-2' selects the endpoint but does not "
                                    "grant access; run from an approved CSDA in-region AWS role/account "
                                    "or request bucket access from CSDA support."
                                )
                            elif error_code in expired_token_codes:
                                advice = " Refresh CSDA S3 credentials and retry the download."
                            else:
                                advice = " Check the network, bucket access, and temporary credential validity."
                            raise RuntimeError(
                                f"S3 download failed for s3://{bucket}/{key}. "
                                f"AWS error {error_code}: {error_message}.{advice}"
                            ) from error
                local_files.append(str(destination))

            manifest_rows.append({
                "site": site_name,
                "selected_year": year,
                "month": MONTH,
                "granule_id": granule_id,
                "title": granule.get("title"),
                "time_start": granule.get("time_start"),
                "s3_uris": ";".join(f"s3://{bucket}/{key}" for bucket, key in links),
                "local_files": ";".join(local_files),
            })
    return manifest_rows


manifest_rows = download_site_granules()
manifest_fields = [
    "site", "selected_year", "month", "granule_id", "title", "time_start",
    "s3_uris", "local_files",
]
with MANIFEST_PATH.open("w", newline="", encoding="utf-8") as manifest_file:
    writer = csv.DictWriter(manifest_file, fieldnames=manifest_fields)
    writer.writeheader()
    writer.writerows(manifest_rows)

print(f"\nDownloaded {len(manifest_rows)} granule(s). Manifest: {MANIFEST_PATH}")
for row in manifest_rows:
    print(f"{row['site']} July {row['selected_year']}: {row['granule_id']}")

RuntimeError: S3 download failed for s3://csda-cumulus-prod-protected-5047/WV02_MSI_L1B___1/2024/185/WV02_20240703155215_10300100FEA6EA00_24JUL03155215-M1BS-508903697020_01_P003/WV02_20240703155215_10300100FEA6EA00_24JUL03155215-M1BS-508903697020_01_P003.ntf. AWS error 403: Forbidden. The AWS principal is not authorized by the CSDA bucket policy. Setting region_name='us-west-2' selects the endpoint but does not grant access; run from an approved CSDA in-region AWS role/account or request bucket access from CSDA support.